# Day 1: PyTorch 핵심 복습 -> HuggingFace로 넘어가기 (해설판)

TODO 빈칸이 모두 채워진 완성 코드와, 각 코드 다음에 '왜 이렇게 동작하는지'를 짧게 설명하는 해설이 추가되어 있습니다. 먼저 practice 노트북에서 직접 풀어본 뒤 이 노트북과 비교해보세요.

실행 환경: Jetson Orin Nano, JetPack 7.2, PyTorch 설치 직후, GPU 1개.

## 1. 텐서 기초: 생성, dtype, device 이동

PyTorch의 모든 데이터는 결국 **텐서(tensor)**입니다. 텐서가 가진 세 가지 핵심 속성을 확인합니다.

- **shape**: 몇 차원이고 각 차원의 크기는 얼마인가
- **dtype**: 원소의 자료형 (`float32`가 기본값. edge device에서는 메모리를 아끼려 `float16`도 자주 씁니다)
- **device**: 이 텐서가 실제로 어느 하드웨어 메모리에 올라가 있는가 (`cpu` 또는 `cuda:0`)

Jetson은 **unified memory** 구조라서 CPU와 GPU가 물리적으로 같은 메모리를 공유합니다. 그래도 PyTorch 입장에서는 여전히 `cpu` 텐서와 `cuda` 텐서를 구분해서 다룹니다 - 어떤 연산 커널(CPU 커널 vs CUDA 커널)을 쓸지가 device로 결정되기 때문입니다.

In [ ]:
import torch

print("torch version:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

x_cpu = torch.tensor([1.0, 2.0, 3.0])
print(x_cpu, x_cpu.dtype, x_cpu.device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_gpu = x_cpu.to(device)
print(x_gpu, x_gpu.dtype, x_gpu.device)

In [ ]:
y = torch.randn(2, 3)          # (2,3) 랜덤 텐서 생성, 기본 dtype=float32
print("dtype:", y.dtype)

y_gpu = y.to(device)            # device 변수에 따라 GPU 또는 CPU로 이동
print("device:", y_gpu.device)

z = y_gpu.cpu().numpy()         # numpy 변환 전에는 반드시 cpu로 되돌려야 함
print(z)

**왜 이렇게 동작하나요?** `torch.randn(2,3)`은 기본적으로 `float32` dtype의 CPU 텐서를 만듭니다. `.to(device)`는 `device`가 `'cuda'`일 때만 실제로 GPU 메모리로 데이터를 복사하고, GPU가 없는 환경에서는 `device`가 `'cpu'`이므로 이 코드가 그대로 에러 없이 동작합니다 (같은 코드로 GPU 유무에 상관없이 돌아가게 만드는 흔한 패턴입니다). `.numpy()`는 CPU 텐서에서만 가능하기 때문에 GPU 텐서는 반드시 `.cpu()`로 먼저 옮겨야 합니다 (그리고 `requires_grad=True`인 텐서라면 `.detach()`도 먼저 필요합니다 - 이 부분은 2번 autograd에서 이어집니다).

### 연습문제 1

서로 다른 device에 있는 텐서끼리는 그냥 더할 수 없습니다. `a`(cpu)와 `b`(cuda)를 더했을 때 나는 에러를 직접 확인하고, device를 맞춰서 고쳐봅니다.

In [ ]:
a = torch.tensor([1.0, 2.0])              # cpu 텐서
b = torch.tensor([3.0, 4.0]).to(device)   # device 텐서

try:
    result = a + b
except RuntimeError as e:
    print("에러 발생:", e)

# 고친 버전: device를 맞춰줌
result = a.to(device) + b
print(result)

**왜 에러가 나나요?** PyTorch의 각 연산 커널은 텐서가 어느 device에 있는지에 따라 서로 다른 구현(C++/CUDA kernel)을 호출합니다. `a`(cpu)와 `b`(cuda)는 서로 다른 메모리 공간에 있다고 간주되기 때문에 (Jetson처럼 실제로는 같은 물리 메모리를 쓰더라도, PyTorch API 레벨에서는 구분됩니다), 자동으로 어느 한쪽을 옮겨주지 않고 명시적으로 `RuntimeError: Expected all tensors to be on the same device` 를 던집니다. 해결책은 항상 '어느 device에서 계산할지'를 정하고 그쪽으로 텐서를 맞춰서 옮기는 것입니다.

## 2. Autograd: 미분을 자동으로 해주는 엔진

`requires_grad=True`로 표시된 텐서에 대해 수행되는 모든 연산을 PyTorch가 기록해뒀다가, `.backward()`가 호출되는 순간 체인룰로 거꾸로 미분값(`.grad`)을 계산합니다. y = x^2 의 미분은 dy/dx = 2x, x=3이면 6이어야 합니다.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

print(f"x = {x.item()}, y = {y.item()}")
print(f"dy/dx (x.grad) = {x.grad.item()}")  # 이론값: 2 * 3 = 6

In [ ]:
# y = x^3 + 2x, dy/dx = 3x^2 + 2, x=2에서 3*4+2 = 14

x2 = torch.tensor(2.0, requires_grad=True)
y2 = x2 ** 3 + 2 * x2
y2.backward()
print("x2.grad =", x2.grad.item())

**왜 14가 나오나요?** y = x^3 + 2x 를 손으로 미분하면 dy/dx = 3x^2 + 2 입니다. x=2를 대입하면 3*4+2=14. PyTorch는 `x2**3 + 2*x2` 라는 연산 그래프(거듭제곱 노드 -> 곱셈 노드 -> 덧셈 노드)를 기록해뒀다가, `backward()` 호출 시 각 노드의 로컬 미분을 체인룰로 곱하고 더해서 이 값을 자동으로 계산합니다. 즉 우리가 미분 공식을 직접 유도하지 않아도 됩니다 - 이게 autograd의 핵심입니다.

### 연습문제 2

벡터 입력 `x3 = [1.0, 2.0, 3.0]`에 대해 `z = (x3**2).sum()`, `z.backward()`를 하면 `x3.grad`는 `2*x3` (`[2,4,6]`)이 되어야 합니다. `backward()`를 한 번 더 호출하면 grad가 어떻게 변하는지 확인합니다.

In [ ]:
x3 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

z = (x3 ** 2).sum()
z.backward()
print("1번째 backward 이후 grad:", x3.grad)  # 기대값: tensor([2., 4., 6.])

z2 = (x3 ** 2).sum()
z2.backward()
print("2번째 backward 이후 grad:", x3.grad)  # 기대값: tensor([4., 8., 12.]) (누적됨)

**왜 grad가 두 배가 되나요?** PyTorch는 기본적으로 `.grad`에 새 미분값을 덮어쓰지 않고 '더합니다'(누적, accumulate). 이는 하나의 파라미터에 대해 여러 번의 forward/backward를 거쳐 grad를 합산해야 하는 경우(예: RNN)를 위한 설계입니다. 하지만 일반적인 학습 루프에서는 매 스텝마다 이전 배치의 grad가 남아있으면 안 되므로, `optimizer.zero_grad()` (또는 텐서 단위로 `x3.grad = None`)로 반드시 초기화해야 합니다. 3번의 학습 루프에서 `optimizer.zero_grad()`를 forward 전에 호출했던 이유가 바로 이것입니다.

## 3. nn.Module로 모델 만들고 학습 루프 돌리기

학습 루프는 어떤 모델이든 forward -> loss -> backward -> optimizer.step (+ zero_grad) 의 반복입니다. 가장 단순한 예로 `y = 3x + 2`를 흉내내는 선형회귀를 학습시킵니다.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
x_train = torch.linspace(-5, 5, 100).unsqueeze(1)
y_train = 3 * x_train + 2 + torch.randn_like(x_train) * 0.5

model = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(200):
    y_pred = model(x_train)
    loss = loss_fn(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss.item():.4f}")

w, b = model.weight.item(), model.bias.item()
print(f"학습된 w={w:.3f}, b={b:.3f}  (목표: w=3, b=2)")

In [ ]:
x_train_gpu = x_train.to(device)
y_train_gpu = y_train.to(device)

model_gpu = nn.Linear(1, 1).to(device)
optimizer_gpu = torch.optim.SGD(model_gpu.parameters(), lr=0.01)

for epoch in range(200):
    y_pred = model_gpu(x_train_gpu)
    loss = loss_fn(y_pred, y_train_gpu)
    optimizer_gpu.zero_grad()
    loss.backward()
    optimizer_gpu.step()

print("최종 loss:", loss.item())
print("model_gpu 파라미터 device:", next(model_gpu.parameters()).device)

**왜 모델도 `.to(device)`가 필요한가요?** `nn.Module`은 내부적으로 `nn.Parameter`(가중치, 편향)를 들고 있는데, `model.to(device)`를 호출하면 이 파라미터들을 전부 해당 device로 이동시킵니다. forward 연산(`model_gpu(x_train_gpu)`)이 성립하려면 입력 텐서와 파라미터가 반드시 같은 device에 있어야 합니다 (아니면 1번에서 본 것과 똑같은 device mismatch 에러가 납니다). 데이터와 모델을 둘 다 옮기는 걸 잊지 않는 것이 실무에서 가장 흔한 실수 포인트입니다.

### 연습문제 3

목표 함수를 `y = -2x + 5`로 바꿔서 모델이 `w≈-2, b≈5`를 찾는지 확인하고, 학습률(lr)을 바꿔가며 수렴 속도를 비교합니다.

In [ ]:
x_train2 = torch.linspace(-5, 5, 100).unsqueeze(1)
y_train2 = -2 * x_train2 + 5 + torch.randn_like(x_train2) * 0.5

model2 = nn.Linear(1, 1)
optimizer2 = torch.optim.SGD(model2.parameters(), lr=0.1)  # lr을 0.1 / 0.01 / 0.0001 등으로 바꿔가며 실험

for epoch in range(200):
    y_pred = model2(x_train2)
    loss = loss_fn(y_pred, y_train2)
    optimizer2.zero_grad()
    loss.backward()
    optimizer2.step()
    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss.item():.4f}")

print(f"w={model2.weight.item():.3f}, b={model2.bias.item():.3f}  (목표: w=-2, b=5)")

**학습률(lr)의 역할:** `optimizer.step()`이 실제로 하는 일은 각 파라미터에 대해 `param = param - lr * param.grad` 입니다. lr이 너무 크면 loss가 진동하거나 발산할 수 있고, 너무 작으면(예: 0.0001) 200 epoch 안에 수렴하지 못해 loss가 거의 줄지 않습니다. 즉 lr은 한 스텝에 얼마나 크게 이동할지를 정하는 스케일 값입니다.

## 4. CPU vs GPU 벤치마크: 행렬곱으로 눈으로 확인하기

CUDA 연산은 비동기이므로 `torch.cuda.synchronize()`로 실제 연산 종료를 기다려야 하고, 첫 CUDA 호출의 워밍업 비용도 벤치마크에서 제외해야 합니다.

In [ ]:
import time

size = 2048
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
c_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = time.time() - start
print(f"CPU matmul({size}x{size}): {cpu_time*1000:.2f} ms")

if torch.cuda.is_available():
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    for _ in range(3):
        _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start = time.time()
    c_gpu = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU matmul({size}x{size}): {gpu_time*1000:.2f} ms  (device: {torch.cuda.get_device_name(0)})")
    print(f"speedup: {cpu_time / gpu_time:.1f}x")
else:
    print("GPU를 사용할 수 없어 비교를 생략합니다.")

In [ ]:
def benchmark_matmul(size):
    a = torch.randn(size, size)
    b = torch.randn(size, size)

    start = time.time()
    _ = torch.matmul(a, b)
    cpu_time = time.time() - start

    a_gpu = a.to(device)
    b_gpu = b.to(device)
    for _ in range(3):
        _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start = time.time()
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start

    return cpu_time, gpu_time

cpu_t, gpu_t = benchmark_matmul(1024)
print(f"1024x1024 -> CPU {cpu_t*1000:.2f}ms, GPU {gpu_t*1000:.2f}ms, speedup {cpu_t/gpu_t:.1f}x")

**synchronize()가 없으면 왜 문제가 되나요?** `torch.matmul(a_gpu, b_gpu)`는 GPU에 작업을 '요청'만 하고 즉시 리턴됩니다(비동기). `torch.cuda.synchronize()`를 호출하지 않으면 `time.time()`이 실제 GPU 연산이 끝나기 전 시각을 재게 되어 GPU가 훨씬 빠른 것처럼 착시가 생깁니다. 워밍업 루프는 CUDA kernel의 최초 컴파일/캐싱 비용을 벤치마크 구간 밖으로 빼내기 위한 것입니다.

### 연습문제 4

여러 크기에 대해 CPU/GPU 시간과 speedup을 표로 출력하고, 작은 크기에서 GPU가 느린 이유를 생각해봅니다.

In [ ]:
sizes = [128, 256, 512, 1024, 2048]

print(f"{'size':>6} | {'CPU(ms)':>10} | {'GPU(ms)':>10} | {'speedup':>8}")
for s in sizes:
    cpu_t, gpu_t = benchmark_matmul(s)
    print(f"{s:>6} | {cpu_t*1000:>10.3f} | {gpu_t*1000:>10.3f} | {cpu_t/gpu_t:>7.2f}x")

**작은 크기에서는 왜 GPU가 안 빠를 수 있나요?** GPU 연산에는 커널 실행(launch) 오버헤드가 있는데, 128x128 같은 작은 행렬곱은 계산량 자체가 적어서 이 오버헤드가 전체 시간에서 차지하는 비중이 커집니다. 반대로 2048x2048처럼 계산량(FLOPs)이 커지면 GPU의 병렬 연산 능력이 오버헤드를 압도하면서 speedup이 뚜렷해집니다. 실무에서 '이 연산을 GPU로 옮길 가치가 있는가'를 판단할 때 항상 이 트레이드오프를 고려해야 합니다.

## 5. HuggingFace Transformers 첫걸음

Transformers(AutoModel/AutoTokenizer), datasets, accelerate(단일/멀티 GPU 실행 래퍼), PEFT(LoRA/QLoRA 구현체), TRL(SFTTrainer, DPO/GRPO)로 이어지는 생태계에서, 오늘은 Transformers로 작은 Qwen 모델을 불러와 추론까지 해봅니다. LoRA는 사전학습 가중치 W는 얼리고 저랭크 ΔW=B·A(r≪d)만 학습하는 방법이라는 점을 기억해두세요.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
).to(device)
model.eval()

prompt = "Jetson Orin Nano에서 딥러닝을 배우는 재미를 한 문장으로 말해줘."
messages = [{"role": "user", "content": prompt}]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("입력 토큰 개수:", inputs["input_ids"].shape[1])

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
my_prompt = "Jetson에서 LLM을 돌리면 좋은 점을 한 문장으로 알려줘."

token_ids = tokenizer.encode(my_prompt)
print("token ids:", token_ids)

decoded = tokenizer.decode(token_ids)
print("decoded:", decoded)

messages2 = [{"role": "user", "content": my_prompt}]
input_text2 = tokenizer.apply_chat_template(messages2, tokenize=False, add_generation_prompt=True)
inputs2 = tokenizer(input_text2, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids2 = model.generate(
        **inputs2,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
    )
print(tokenizer.decode(output_ids2[0], skip_special_tokens=True))

**encode/decode와 chat template의 관계:** `tokenizer.encode`는 순수하게 '문자열 -> 토큰 id 리스트' 변환만 하고, 모델이 기대하는 대화 형식(system/user/assistant role, 특수 토큰 등)은 전혀 신경 쓰지 않습니다. Instruct/Chat 모델은 학습할 때부터 특정 포맷을 봤기 때문에, `apply_chat_template`으로 그 포맷을 맞춰주지 않으면 모델이 이상하게 답할 수 있습니다. `do_sample=True`와 `temperature`는 다음 토큰을 고를 때 확률분포에서 얼마나 무작위성을 줄지 조절하는 값이고, `do_sample=False`(greedy)는 항상 가장 확률 높은 토큰만 골라서 같은 입력에 항상 같은 출력이 나옵니다.

### 연습문제 5

CPU/GPU 생성 시간을 비교하고, LoRA가 왜 적은 자원으로 fine-tuning을 가능하게 하는지 설명해봅니다.

In [ ]:
import time

def generate_and_time(model, inputs, max_new_tokens=20):
    dev = next(model.parameters()).device
    if dev.type == "cuda":
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=5, do_sample=False)  # 워밍업
        torch.cuda.synchronize()

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start
    return elapsed, output_ids

# GPU에서 측정
inputs_gpu = tokenizer(input_text, return_tensors="pt").to(device)
gpu_time, _ = generate_and_time(model, inputs_gpu, max_new_tokens=20)
print(f"GPU 생성 시간: {gpu_time*1000:.1f} ms")

# CPU로 옮겨서 측정 (모델이 작아도 CPU에서는 느릴 수 있습니다)
model_cpu = model.to("cpu")
inputs_cpu = tokenizer(input_text, return_tensors="pt")
cpu_time, _ = generate_and_time(model_cpu, inputs_cpu, max_new_tokens=20)
print(f"CPU 생성 시간: {cpu_time*1000:.1f} ms")

# 다시 GPU로 복구
model = model.to(device)

print(f"speedup: {cpu_time/gpu_time:.1f}x")

**LLM 추론에서 GPU가 특히 중요한 이유:** `model.generate`는 토큰을 하나씩 순차적으로 만들어내는데(autoregressive), 매 스텝마다 모델 전체를 한 번씩 통과시켜야 합니다. 0.5B급의 작은 모델도 파라미터가 수억 개라 CPU에서는 이 반복이 눈에 띄게 느립니다.

그리고 이 모델에 LoRA를 적용한다고 하면, `nn.Linear`의 원래 가중치 W(예: 어텐션의 Q/K/V/O 프로젝션 행렬)는 그대로 얼려두고, 그 옆에 훨씬 작은 두 행렬 B(d×r)와 A(r×d)를 새로 붙여서 ΔW=B·A 만 학습합니다. r을 d보다 훨씬 작게 잡기 때문에(r≪d) 학습해야 할 파라미터 수와 옵티마이저 상태(Adam이면 파라미터당 추가 버퍼 2개)가 극적으로 줄어들어, Jetson처럼 메모리가 제한된 환경에서도 전체 fine-tuning보다 훨씬 적은 자원으로 학습이 가능해집니다.

## 마무리

텐서 -> autograd -> nn.Module 학습 루프 -> device/속도 -> HuggingFace Transformers, 이 순서로 PyTorch의 '바닥'부터 HuggingFace가 감싸주는 '위층'까지 한 번 훑었습니다. 아래 면접 Q&A로 정리해보세요.

## 면접 Q&A — 이 노트북으로 답할 수 있게 된 질문

## 면접 대비 Q&A (오늘 실습 기반)

이 노트북을 마치면 아래 질문들에 대해, 모르는 걸 아는 척하지 않고 오늘 직접 손으로 확인한 범위 안에서 정직하게 답할 수 있습니다.

### Q1. PyTorch의 autograd는 어떻게 동작하나요? `requires_grad`와 `backward()`를 설명해주세요.

**A.** `requires_grad=True`로 표시한 텐서에 대해 수행되는 연산들을 PyTorch가 동적으로 계산 그래프(computational graph)로 기록합니다. `.backward()`를 호출하면 이 그래프를 역방향으로 순회하면서 체인룰(chain rule)로 각 텐서의 미분값을 계산해 `.grad`에 채워줍니다. 오늘 `y=x**2`, `y=x**3+2*x` 같은 스칼라 예제와, 벡터에 `.sum()`을 씌운 뒤 `backward()`를 두 번 호출해서 `.grad`가 덮어써지지 않고 누적된다는 것까지 직접 확인했습니다. 이 누적 특성 때문에 학습 루프에서는 매 스텝 `optimizer.zero_grad()`로 이전 grad를 반드시 지워줘야 합니다.

### Q2. GPU 벤치마크를 할 때 주의해야 할 점은 무엇인가요?

**A.** 두 가지입니다. 첫째, CUDA 연산은 비동기라서 `torch.matmul(...)` 호출이 리턴돼도 실제 GPU 연산은 아직 끝나지 않았을 수 있습니다 - `torch.cuda.synchronize()`로 실제 종료 시점까지 기다린 뒤 시간을 재야 합니다. 둘째, 첫 CUDA 호출에는 컨텍스트 초기화/커널 컴파일 같은 워밍업 비용이 섞여 들어가므로 벤치마크 전에 워밍업을 몇 번 돌려줘야 합니다. 오늘 실습에서 크기를 128부터 2048까지 바꿔가며 재보니, 작은 행렬에서는 커널 실행 오버헤드가 실제 연산량보다 커서 GPU 이득이 크지 않고, 행렬이 커질수록 speedup이 뚜렷해지는 것도 직접 확인했습니다.

### Q3. LoRA가 왜 동작하는지 설명해주실 수 있나요?

**A.** 사전학습된 가중치 W는 그대로 동결(freeze)하고, 그 옆에 저랭크 행렬 B(d×r)와 A(r×d)를 추가해서 ΔW = B·A 형태의 아주 작은 파라미터만 학습하는 방법입니다. r을 d보다 훨씬 작게(r≪d) 잡는데, 이게 가능한 이유는 fine-tuning 과정에서 실제로 일어나는 가중치 변화가 사실상 저랭크라는 관찰 때문입니다. 저는 아직 Transformer의 attention을 밑바닥부터 구현해본 건 아니지만, 오늘 `nn.Linear`가 어떤 파라미터를 학습 대상으로 갖는지, 학습 루프가 그 파라미터의 grad를 어떻게 업데이트하는지를 직접 만들어봤기 때문에, LoRA가 '기존 `nn.Linear`의 W는 grad 계산 대상에서 빼고, 새로 붙인 훨씬 작은 두 행렬만 학습 대상으로 바꾸는 것'이라는 구조는 정확히 이해하고 있습니다. 그 결과 학습해야 할 파라미터 수와 옵티마이저 상태가 크게 줄어서 메모리/연산 비용이 절감됩니다.

### Q4. HuggingFace 생태계에서 Transformers, datasets, accelerate, PEFT, TRL은 각각 어떤 역할을 하나요?

**A.** Transformers는 `AutoModel`/`AutoTokenizer`처럼 다양한 모델과 토크나이저를 통일된 인터페이스로 불러오는 핵심 라이브러리입니다. datasets는 학습/평가용 데이터셋을 다운로드하고 전처리하는 역할을 합니다. accelerate는 같은 학습 코드를 단일 GPU든 멀티 GPU/분산 환경이든 거의 수정 없이 돌릴 수 있게 해주는 실행 래퍼입니다. PEFT는 LoRA, QLoRA처럼 파라미터 일부만 학습하는 기법들의 구현체이고, TRL은 `SFTTrainer`나 DPO/GRPO 트레이너처럼 LLM을 지시-따르기나 선호도 학습 방식으로 미세조정할 때 쓰는 도구입니다. 오늘은 이 중 Transformers만 직접 실습해봤고, 나머지는 이번 주 안에 개념과 최소 사용법 위주로 더 볼 계획입니다.

### Q5. Jetson 같은 edge 디바이스에서 모델을 서빙할 때 고려해야 할 점은 무엇인가요?

**A.** 아직 실제 프로덕션 서빙까지 경험해본 것은 아니지만, 오늘 실습 범위에서 확인한 것을 말씀드리면: 첫째, Jetson은 CPU와 GPU가 같은 물리 메모리를 쓰는 unified memory 구조라 대형 GPU 서버처럼 별도 VRAM을 늘릴 수 없고, 그만큼 모델 크기(파라미터 수)와 정밀도(dtype) 선택이 더 중요합니다 - 그래서 오늘도 `float16`으로 작은 모델(0.5B)을 올렸습니다. 둘째, 첫 추론에는 CUDA 워밍업 비용이 있으니 실제 서빙에서는 서버 기동 직후 워밍업 요청을 미리 흘려주는 것이 좋습니다. 셋째, LLM 추론은 토큰을 하나씩 순차 생성하는 autoregressive 구조라 배치 크기 1의 실시간 응답 시간이 중요한 지표가 되는데, 이 부분과 int8 양자화 등 더 깊은 최적화는 아직 제가 직접 실험해보지 못했고 앞으로 공부할 부분입니다.
